# COG route on Databricks: smoke test

Runs the admin-2 COG route (`pipeline/steps/s01b.py`) for **3 small countries × 3 hazards**
and checks the results against the same run on a laptop. It reads the WorldPop and hazard COGs
from Azure (anonymous HTTPS), reads the admin-2 boundaries from Azure too
(`public/boundaries/adm2/<adm0_ucode>.parquet`), uses **no Earth Engine**, and fits easily on a
single-node cluster (16 GB, 4 cores): the largest population grid here is about 3,000 × 2,500 pixels.

| | |
|---|---|
| Countries | Lesotho, Eswatini, Rwanda |
| Hazards | air pollution (PM2.5), earthquake (PGA > 0.09 g), landslides (> 0.1) |
| Expected time | about 1–3 minutes, mostly package installation |
| Writes | about 7 MB of boundaries, 0.14 GB of population files (deleted after each country) and 1 MB of CSVs, on the driver's local disk |

## Setup

1. Upload `gchd_cog_smoke.zip` to a Unity Catalog volume or to DBFS, and import this notebook.
2. Set `BUNDLE_ZIP` below to the uploaded path, for example `/Volumes/<catalog>/<schema>/<volume>/gchd_cog_smoke.zip`
   or `/dbfs/FileStore/gchd_cog_smoke.zip`.
3. Attach the cluster and run all cells. The install cell restarts Python once.

If the repository is available as a Databricks Git folder instead, leave `BUNDLE_ZIP = None` and
open this notebook from `notebooks/pipeline/`.

In [ ]:
# Install the open-source packages (once per cluster session; restarts Python)
%pip install -q "exactextract==0.3.0" "rasterio>=1.3" geopandas pyarrow "pydantic>=2" earthengine-api
dbutils.library.restartPython()

In [ ]:
# ============================================================
# Configuration
# ============================================================
BUNDLE_ZIP = None                 # e.g. "/Volumes/<catalog>/<schema>/<volume>/gchd_cog_smoke.zip"
COUNTRIES  = ["LSO_V1", "SWZ_V1", "RWA_V1"]
HAZARDS    = ["air_pollution_pm25", "earthquake_pga_0p09", "landslides_0p1"]
POPULATION = "download"           # cache the population COGs on local disk first (~0.14 GB here), so compute_s
                                  # is computation only; "remote" reads them over the network during it
WORKERS    = 2                    # countries at once; 2 is plenty on 4 cores
OUT_DIR    = "/tmp/gchd_cog_smoke"
CACHE_DIR  = "/tmp/gchd_cog_cache"   # boundaries/ and worldpop/; emptied below so the boundaries come from Azure

In [ ]:
# Locate the code: the repository (Git folder or local), or the unzipped bundle
import os, sys, zipfile
from pathlib import Path
here = [p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pipeline" / "settings.py").is_file()]
if here and not BUNDLE_ZIP:
    CODE_ROOT = here[0]
else:
    CODE_ROOT = Path("/tmp/gchd_cog_bundle")  # local disk of the driver
    with zipfile.ZipFile(BUNDLE_ZIP) as z:
        z.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("code:", CODE_ROOT)

In [ ]:
# Machine and library versions (for the timing comparison)
import platform, rasterio, exactextract
mem_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9 if hasattr(os, "sysconf") else float("nan")
print(f"cores {os.cpu_count()} | memory {mem_gb:.0f} GB | Python {platform.python_version()} | "
      f"rasterio {rasterio.__version__} (GDAL {rasterio.__gdal_version__}) | exactextract {exactextract.__version__}")
try:
    print("cluster region:", spark.conf.get("spark.databricks.clusterUsageTags.region"))
except Exception:
    pass

In [ ]:
# Admin-2 boundaries from Azure: one GeoParquet per country, keyed by adm2_ucode
import shutil, time, requests
from pipeline.settings import Settings
from pipeline.steps import s01b
settings = Settings.from_env(storage_root=OUT_DIR, process_countries=COUNTRIES, process_hazards=HAZARDS)
shutil.rmtree(Path(CACHE_DIR) / "boundaries", ignore_errors=True)
print("source:", settings.boundaries_url)
for u in COUNTRIES:
    url = f"{settings.boundaries_url}/{u}.parquet"
    head = requests.head(url, timeout=60)
    assert head.status_code == 200, f"{u}: HTTP {head.status_code} at {url} (s01b would fall back to Earth Engine)"
    t = time.time()
    g = s01b.boundaries(settings, u, Path(CACHE_DIR))
    assert g.adm2_ucode.notna().all() and g.adm2_ucode.is_unique, f"{u}: adm2_ucode missing or duplicated"
    print(f"  {u}: {len(g)} units, {int(head.headers['Content-Length']) / 1e6:.1f} MB in {time.time() - t:.1f} s, "
          f"CRS {g.crs.to_string()}, columns {len(g.columns)}")

In [ ]:
# Run (boundaries are now cached locally; population and hazards are read from Azure)
result = s01b.run(settings, cache_dir=str(CACHE_DIR), population=POPULATION, workers=WORKERS, max_cache_gb=2)
result

In [ ]:
# Timing per country: compute_s is the computation alone; hazard_read_s and population_s are data reads
import pandas as pd
from pipeline import io_paths
timing = pd.read_csv(Path(io_paths.manifest_root(settings)) / "s01b_timing.csv")
timing

In [ ]:
# Same results as the laptop run?
REFERENCE = {
 "LSO_V1|air_pollution_pm25": [
  2350212,
  954962,
  27.659311,
  78
 ],
 "LSO_V1|earthquake_pga_0p09": [
  0,
  0,
  0.034532,
  78
 ],
 "LSO_V1|landslides_0p1": [
  0,
  0,
  0.00028,
  78
 ],
 "SWZ_V1|air_pollution_pm25": [
  1248517,
  493099,
  17.8138,
  59
 ],
 "SWZ_V1|earthquake_pga_0p09": [
  0,
  0,
  0.027034,
  59
 ],
 "SWZ_V1|landslides_0p1": [
  0,
  0,
  0.000461,
  59
 ],
 "RWA_V1|air_pollution_pm25": [
  14354782,
  6306027,
  42.267996,
  31
 ],
 "RWA_V1|earthquake_pga_0p09": [
  8658511,
  3867695,
  0.143891,
  31
 ],
 "RWA_V1|landslides_0p1": [
  0,
  0,
  0.005021,
  31
 ]
}
rows = []
for key, (exp, u18, mean, units) in REFERENCE.items():
    u, h = key.split("|")
    if u not in COUNTRIES or h not in HAZARDS:
        continue
    d = pd.read_csv(io_paths.base_partial_csv(settings, h, u))
    got = (round(d.exp_total.sum()), round(d.exp_under_18_total.sum()), round(float(d.hazs_mean.mean()), 6), len(d))
    ok = got[0] == exp and got[1] == u18 and abs(got[2] - mean) <= 1e-6 * max(1, abs(mean)) and got[3] == units
    rows.append(dict(country=u, hazard=h, exposed=got[0], expected=exp, under_18=got[1], expected_u18=u18,
                     units=got[3], match="yes" if ok else "NO"))
check = pd.DataFrame(rows)
print("all match" if (check.match == "yes").all() else "MISMATCH, see rows marked NO")
check

## Next

For more countries or hazards, extend `COUNTRIES` / `HAZARDS` (any of the 14 codes in
`pipeline/cog_zonal.py`, `COG_LAYERS`). A country's admin-2 boundaries are downloaded on first use
from `public/boundaries/adm2/` (setting `boundaries_url`), so no Earth Engine credentials are needed.
`WORKERS` up to the number of cores;
large countries (Ethiopia, Kenya) need several GB of memory each.